In [27]:
import pandas as pd
import json
import numpy as np

DATA_LIMIT = 2000

# -----------------------------
# Recursively convert to JSON-safe Python objects
# -----------------------------

def make_json_serializable(obj):

    # NumPy array
    if isinstance(obj, np.ndarray):
        return [
            make_json_serializable(x)
            for x in obj.tolist()
        ]

    # NumPy scalar
    if isinstance(obj, np.generic):
        return obj.item()

    # Dictionary
    if isinstance(obj, dict):
        return {
            str(k): make_json_serializable(v)
            for k, v in obj.items()
        }

    # List / tuple
    if isinstance(obj, (list, tuple)):
        return [
            make_json_serializable(x)
            for x in obj
        ]

    # Pandas NA / NaN
    if pd.isna(obj):
        return None

    return obj


# -----------------------------
# Read parquet
# -----------------------------

df = pd.read_parquet(
    "ms_marco_validation.parquet"
)

# Take first 2000
records = df.head(DATA_LIMIT).to_dict(
    orient="records"
)

# Convert everything recursively
records = make_json_serializable(records)


# -----------------------------
# Final JSON
# -----------------------------

ms_marco_data = {
    "dataset_name": "ms_marco",
    "dataset_link":
        "https://huggingface.co/datasets/microsoft/ms_marco/viewer/v1.1/validation",
    "total_documents": len(records),
    "documents": records
}


# -----------------------------
# Save
# -----------------------------

with open(
    "ms_marco_top2k.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        ms_marco_data,
        f,
        indent=2,
        ensure_ascii=False,
        allow_nan=False
    )

print(
    f"Created ms_marco_topk.json "
    f"with {len(records)} documents"
)

Created ms_marco_topk.json with 2000 documents


# build Qdrant vector db

In [4]:
import ast
import uuid

from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

from qdrant_client import QdrantClient, models

In [29]:
with open("ms_marco_top2k.json",'r') as f:
    documents = json.load(f)

In [30]:
documents = documents['documents']

In [5]:
COLLECTION_NAME = "ms_marco_hybrid"

client = QdrantClient(
    url="http://localhost:6333"
)

# Dense
dense_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Sparse BM25
sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)

Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 11.49it/s]


In [6]:
DENSE_DIM = dense_model.get_sentence_embedding_dimension()

print(DENSE_DIM)

384


/tmp/ipykernel_31750/3050639698.py:1: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  DENSE_DIM = dense_model.get_sentence_embedding_dimension()


In [33]:
def create_collection():

    # Delete if already exists
    if client.collection_exists(COLLECTION_NAME):
        client.delete_collection(COLLECTION_NAME)

    client.create_collection(
        collection_name=COLLECTION_NAME,

        vectors_config={
            "dense": models.VectorParams(
                size=DENSE_DIM,
                distance=models.Distance.COSINE
            )
        },

        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )
        }
    )

    print(f"Created collection: {COLLECTION_NAME}")


create_collection()

Created collection: ms_marco_hybrid


In [35]:
def parse_list_string(value):

    if isinstance(value, list):
        return value

    return ast.literal_eval(value)

def create_points(documents):

    points = []

    for document in documents:

        query_id = document["query_id"]

        passages = parse_list_string(
            document["passages"]["passage_text"]
        )

        urls = parse_list_string(
            document["passages"]["url"]
        )


        query = document["query"]

        for chunk_idx, text in enumerate(passages):

            chunk_id = f"{query_id}_{chunk_idx}"

            # -------------------------
            # Dense embedding
            # -------------------------

            dense_vector = dense_model.encode(
                text,
                normalize_embeddings=True
            ).tolist()

            # -------------------------
            # Sparse BM25 embedding
            # -------------------------

            sparse_vector = list(
                sparse_model.embed([text])
            )[0]

            sparse = models.SparseVector(
                indices=sparse_vector.indices.tolist(),
                values=sparse_vector.values.tolist()
            )

            # -------------------------
            # Metadata
            # -------------------------

            payload = {
                "query_id": query_id,
                "chunk_id": chunk_id,
                "query": query,
                "text": text,
                "answer": document["answers"],
                "url": urls[chunk_idx],
                "query_type": document.get("query_type"),
                "wellFormedAnswers": document.get(
                    "wellFormedAnswers"
                )
            }

            # -------------------------
            # Qdrant point
            # -------------------------

            points.append(
                models.PointStruct(
                    id=str(uuid.uuid4()),

                    vector={
                        "dense": dense_vector,
                        "sparse": sparse
                    },

                    payload=payload
                )
            )

    return points


points = create_points(documents)

print("Total points:", len(points))

Total points: 16396


In [37]:
BATCH_SIZE = 100

for i in range(0, len(points), BATCH_SIZE):

    batch = points[i:i + BATCH_SIZE]

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch,
        wait=True
    )

    print(
        f"Uploaded {min(i + BATCH_SIZE, len(points))}/{len(points)}"
    )

Uploaded 100/16396
Uploaded 200/16396
Uploaded 300/16396
Uploaded 400/16396
Uploaded 500/16396
Uploaded 600/16396
Uploaded 700/16396
Uploaded 800/16396
Uploaded 900/16396
Uploaded 1000/16396
Uploaded 1100/16396
Uploaded 1200/16396
Uploaded 1300/16396
Uploaded 1400/16396
Uploaded 1500/16396
Uploaded 1600/16396
Uploaded 1700/16396
Uploaded 1800/16396
Uploaded 1900/16396
Uploaded 2000/16396
Uploaded 2100/16396
Uploaded 2200/16396
Uploaded 2300/16396
Uploaded 2400/16396
Uploaded 2500/16396
Uploaded 2600/16396
Uploaded 2700/16396
Uploaded 2800/16396
Uploaded 2900/16396
Uploaded 3000/16396
Uploaded 3100/16396
Uploaded 3200/16396
Uploaded 3300/16396
Uploaded 3400/16396
Uploaded 3500/16396
Uploaded 3600/16396
Uploaded 3700/16396
Uploaded 3800/16396
Uploaded 3900/16396
Uploaded 4000/16396
Uploaded 4100/16396
Uploaded 4200/16396
Uploaded 4300/16396
Uploaded 4400/16396
Uploaded 4500/16396
Uploaded 4600/16396
Uploaded 4700/16396
Uploaded 4800/16396
Uploaded 4900/16396
Uploaded 5000/16396
Uploaded 

# Search

In [12]:
from flashrank import Ranker, RerankRequest

ranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2"
)

INFO:flashrank.Ranker:Downloading ms-marco-MiniLM-L-12-v2...
ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:06<00:00, 3.35MiB/s]


In [ ]:
def apply_reranker(
    query,
    results,
    limit
):

    passages = []

    for result in results:
        passages.append({
            "chunk_id": result.payload["chunk_id"],
            "text": result.payload["text"]
        })

    rerank_request = RerankRequest(
        query=query,
        passages=passages
    )

    reranked = ranker.rerank(rerank_request)

    # FlashRank returns dictionaries sorted by relevance
    reranked = reranked[:limit]

    return reranked

In [ ]:
def dense_search(
    query: str,
    limit: int = 5,
):

    dense_vector = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    results = new_client.query_points(
        collection_name=COLLECTION_NAME,

        query=dense_vector,

        using="dense",

        limit=limit,

        with_payload=True
    )

    return results.points

In [ ]:
results = dense_search(
    "does poppy seed mean mustard seeds",
    limit=5,
)

for result in results:

    print(
        result.score,
        result.payload["query_id"],
        result.payload["chunk_id"]
    )

Batches: 100%|██████████| 1/1 [00:00<00:00, 100.55it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"


id='1ee7deb2-a76e-4c3c-9dce-b7773fd138d0' version=39 score=0.71390283 payload={'query_id': 9845, 'chunk_id': '9845_2', 'query': 'does poppy seed mean mustard seeds', 'text': 'The noun POPPY SEED has 1 sense: 1. small grey seed of a poppy flower; used whole or ground in baked items. Familiarity information: POPPY SEED used as a noun is very rare. ', 'answer': ['No'], 'url': 'http://www.audioenglish.org/dictionary/poppy_seed.htm', 'query_type': 'description', 'wellFormedAnswers': []} vector=None shard_key=None order_value=None
id='7f8958c8-4414-46fa-a485-9336a3b96b90' version=164 score=0.6851644 payload={'query_id': 9845, 'chunk_id': '9845_5', 'query': 'does poppy seed mean mustard seeds', 'text': 'Poppy seed is an oilseed obtained from the opium poppy (Papaver somniferum). The tiny kidney-shaped seeds have been harvested from dried seed pods by various civilizations for thousands of years. The seeds are used, whole or ground, as an ingredient in many foods, and they are pressed to yield

KeyError: 'query_id'

In [21]:
def hybrid_search(
    query: str,
    limit: int = 5,
    candidate_limit: int = 20
):

    # -------------------------
    # Dense query embedding
    # -------------------------

    dense_vector = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # -------------------------
    # Sparse BM25 query
    # -------------------------

    sparse_vector = list(
        sparse_model.embed([query])
    )[0]

    sparse_query = models.SparseVector(
        indices=sparse_vector.indices.tolist(),
        values=sparse_vector.values.tolist()
    )

    # -------------------------
    # Qdrant hybrid search
    # -------------------------

    results = new_client.query_points(
        collection_name=COLLECTION_NAME,

        prefetch=[
            models.Prefetch(
                query=dense_vector,
                using="dense",
                limit=candidate_limit
            ),

            models.Prefetch(
                query=sparse_query,
                using="sparse",
                limit=candidate_limit
            )
        ],

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),

        limit=limit,

        with_payload=True
    )

    return results.points

In [ ]:
def print_semantic_and_hybrid_search(query:str):
    hybrid_results = hybrid_search(
        query,
        limit=10,
        candidate_limit=20
    )


    semantic_results = dense_search(
        query,
        limit=10
    )

    print("==Semantic Result==\n\n")
    for result in semantic_results:

        print(
            result.score,
            result.payload["query_id"],
            result.payload["chunk_id"],
            # result.payload["text"]
        )


    print("\n\n==Hybrid Result==\n\n")

    for result in hybrid_results:

        print(
            result.score,
            result.payload["query_id"],
            result.payload["chunk_id"],
            # result.payload["text"]
        )

Hybrid Search query

11442
 - What is 74ABT162244?
 - What is PCA9544A?
 - What is DN-44?
 - What happened on July 5, 2011

10918
 - What is $460 / Ln. Ft.?

10751
 - What does Ctrl-X do?
  

10826
- What happened in February 1935?

11441
  - What was the C$63,136 salary associated with
  - What was the hourly wage of $34.15 associated with?

reranking

11441
- What compensation can someone expect when entering the accounting profession?

9670
- In legal terminology, what does the abbreviation LOP indicate when a case is closed because of inactivity?

9665
- What distinguishes stock from broth in terms of preparation and flavor intensity?
- Why might someone reduce the amount of salt in a recipe when replacing water with a prepared stock product?

In [23]:
query = r"If someone is comparing a do-it-yourself basement project with a packaged finishing system, what cost ranges should they consider, and which option is described as including insulated wall panels and lighting?"
print_semantic_and_hybrid_search(query)


==Semantic Result==


0.74730384 9658 9658_3
0.6921941 9658 9658_2
0.63287663 9658 9658_0
0.6157731 10571 10571_3
0.5746965 9658 9658_5
0.5588244 9658 9658_1
0.5432924 11641 11641_4
0.5112865 10896 10896_4
0.5082063 10896 10896_1
0.49236184 11512 11512_3


==Hybrid Result==


1.0 9658 9658_3
0.6666667 9658 9658_2
0.39285713 9658 9658_0
0.36666667 9658 9658_5
0.25882354 10571 10571_3
0.25 11075 11075_5
0.22916667 10218 10218_5
0.21111111 10896 10896_1
0.1904762 9658 9658_1
0.13025211 11187 11187_4


In [30]:
result = {
    "documents":[]
}
with open("ms_marco_top2k.json",'r') as f:
    data = json.load(f)
    for count,doc in enumerate(data['documents']):
        tmp_doc = {}
        tmp_doc['query'] = doc['query']
        tmp_doc['expected_answer'] = doc['answers']
        tmp_doc['difficulty'] = "normal"
        query_ids = doc['query_id']
        tmp_doc['query_ids']=query_ids
        indices = [index for index, value in enumerate(doc['passages']['is_selected']) if value == 1]
        tmp_doc['chunk_ids']=[]
        for chunk_id in indices:
            tmp_doc['chunk_ids'].append(f"{query_ids}_{chunk_id}")

        result['documents'].append(tmp_doc)

        if count >23:
            break

with open("data/selected_questions/selected_normal_question.json",'w') as f:
    json.dump(result,f)
# def check_50_percent_overlap(l1, l2):
#     # Convert lists to sets to easily find the common elements
#     set1, set2 = set(l1), set(l2)
    
#     # Find the intersection (elements in both sets)
#     common_elements = set1.intersection(set2)
    
#     # Calculate the percentage of common elements. 
#     # Here, we check if the overlap is >= 50% of the smaller list.
#     # You can change the denominator to len(set1) if you want to check strictly against list 1.
#     smaller_list_length = min(len(set1), len(set2))
    
#     if smaller_list_length == 0:
#         return False # Avoid division by zero if lists are empty
        
#     overlap_percentage = (len(common_elements) / smaller_list_length) * 100
    
#     is_50_percent = overlap_percentage >= 50.0
    
#     return is_50_percent, overlap_percentage, common_elements

# question_which_can_be_select = []
# with open("complex_multi_queryid.json",'r') as f:
#     data = json.load(f)
#     for quest in data['questions']:
#         hybrid_results = hybrid_search(
#                 quest['question'],
#                 limit=10,
#                 candidate_limit=20
#             )
#         chunk_ids_retrived = []
#         chunk_ids_expected = quest['chunk_ids']
#         for result in hybrid_results:
#             chunk_ids_retrived.append(result.payload["chunk_id"])
#         is_overlap, percentage, shared = check_50_percent_overlap(chunk_ids_retrived,chunk_ids_expected)
#         if is_overlap:
#             question_which_can_be_select.append({"question_doc":quest,
#                                 "is_overlap":is_overlap,
#                                 "percentage":percentage,
#                                 "shared":shared,
#                                 "chunk_ids_retrived":chunk_ids_retrived
#             })

In [44]:
import json

# Replace these with the actual names of your 3 JSON files
file_names = ['data/selected_questions/selected_normal_question.json','data/selected_questions/selected_hybrid_question.json','data/selected_questions/selected_complex_questions.json' ]

merged_documents = []

# Loop through each file and extract the documents
for file_name in file_names:
    try:
        with open(file_name, 'r', encoding='utf-8') as f:
            data = json.load(f)
            
            # Check if "documents" key exists and extend the merged list
            if "documents" in data:
                merged_documents.extend(data["documents"])
            else:
                print(f"Warning: 'documents' key not found in {file_name}")
                
    except FileNotFoundError:
        print(f"Error: {file_name} not found. Please check the file name.")
    except json.JSONDecodeError:
        print(f"Error: {file_name} is not a valid JSON file.")

# Create the final structure
final_data = {
    "documents": merged_documents
}

# Save the merged data into a new JSON file
output_filename = 'data/selected_questions/final_merged.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    # indent=4 makes the JSON file human-readable and formatted nicely
    json.dump(final_data, f, indent=4)

print(f"Successfully merged {len(merged_documents)} documents into {output_filename}")

Successfully merged 42 documents into data/selected_questions/final_merged.json


# Evaluate

In [15]:
import pandas as pd


def precision_at_k(retrieved_ids, expected_ids, k):
    retrieved_ids = retrieved_ids[:k]
    expected_ids = set(expected_ids)

    relevant = sum(
        chunk_id in expected_ids
        for chunk_id in retrieved_ids
    )

    return relevant / k


def recall_at_k(retrieved_ids, expected_ids, k):
    retrieved_ids = set(retrieved_ids[:k])
    expected_ids = set(expected_ids)

    relevant = len(retrieved_ids & expected_ids)

    return relevant / len(expected_ids)


def evaluate_retriever(
    search_function,
    golden_data,
    top_k=(5, 10),
    name="dense"
):

    results = []

    max_k = max(top_k)
    try:
        for item in golden_data:

            query = item["query"]
            expected_ids = item["chunk_ids"]

            # Search only once with maximum K
            search_results = search_function(
                query,
                limit=max_k
            )

            retrieved_ids = [
                result.payload["chunk_id"]
                for result in search_results
            ]

            for k in top_k:

                results.append({
                    "query_id": item["query_ids"],
                    "query": query,
                    "retriever": name,
                    "k": k,

                    "precision": precision_at_k(
                        retrieved_ids,
                        expected_ids,
                        k
                    ),

                    "recall": recall_at_k(
                        retrieved_ids,
                        expected_ids,
                        k
                    )
                })
    except Exception as e:
        import traceback
        traceback.print_exc()

    return pd.DataFrame(results)

In [16]:
import json

with open("data/selected_questions/final_merged.json", "r") as f:
    golden_data = json.load(f)["documents"]

In [22]:
dense_results = evaluate_retriever(
    dense_search,
    golden_data,
    top_k=(5,10),
    name="Dense"
)

hybrid_results = evaluate_retriever(
    hybrid_search,
    golden_data,
    top_k=(5,10),
    name="Hybrid"
)

all_results = pd.concat(
    [dense_results, hybrid_results],
    ignore_index=True
)


summary = (
    all_results
    .groupby(["retriever", "k"])[
        ["precision", "recall"]
    ]
    .mean()
    .reset_index()
)

print(summary)

Batches: 100%|██████████| 1/1 [00:00<00:00, 24.18it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 75.91it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 83.32it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 101.27it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 82.84it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00, 54.90it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"
Bat

  retriever   k  precision    recall
0     Dense   5     0.1850  0.664167
1     Dense  10     0.1275  0.770000
2    Hybrid   5     0.2100  0.768333
3    Hybrid  10     0.1475  0.951250


## with reranker

In [29]:
# Reranked
from functools import partial

dense_reranker_results = evaluate_retriever(
    partial(
        dense_search,
        rerank=True,
        limit=10
    ),
    golden_data,
    top_k=(5, 10),
    name="Dense + Reranker"
)

Batches: 100%|██████████| 1/1 [00:00<00:00,  8.84it/s]
INFO:httpx:HTTP Request: POST http://localhost:6333/collections/ms_marco_hybrid/points/query "HTTP/1.1 200 OK"


('points', [ScoredPoint(id='3d7e9506-3bd1-4dd6-baad-359e9d9db958', version=79, score=0.7814467, payload={'query_id': 9652, 'chunk_id': '9652_3', 'query': 'walgreens store sales average', 'text': 'th store in 1984, reaching $4 billion in sales in 1987, and $5 billion two years later. Walgreens ended the 1980s with 1,484 stores, $5.3 billion in revenues and $154 million in profits. However, profit margins remained just below 3 percent of sales, and returns on assets of less than 10 percent.', 'answer': ['Approximately $15,000 per year.'], 'url': 'http://www.babson.edu/executive-education/thought-leadership/retailing/Documents/walgreens-strategic-evolution.pdf', 'query_type': 'numeric', 'wellFormedAnswers': []}, vector=None, shard_key=None, order_value=None), ScoredPoint(id='79f96e16-6f71-48c4-88b4-38450f0aed51', version=157, score=0.77897435, payload={'query_id': 9652, 'chunk_id': '9652_4', 'query': 'walgreens store sales average', 'text': 'The number of Walgreen stores has risen from 5,

Traceback (most recent call last):
  File "/tmp/ipykernel_31750/1053180344.py", line 42, in evaluate_retriever
    search_results = search_function(
        query,
        limit=max_k
    )
  File "/tmp/ipykernel_31750/2563345709.py", line 26, in dense_search
    results = apply_reranker(
        query=query,
        results=results,
        limit=limit
    )
  File "/tmp/ipykernel_31750/3724113833.py", line 12, in apply_reranker
    "id": result.points.payload["chunk_id"],
          ^^^^^^^^^^^^^
AttributeError: 'tuple' object has no attribute 'points'


# save

In [56]:
import json
import gzip

from qdrant_client import models

def to_list(value):
    return value.tolist() if hasattr(value, "tolist") else list(value)


def make_json_serializable(record):
    record = dict(record)

    if "vector" in record:
        vector = dict(record["vector"])

        if "sparse" in vector:
            sparse = vector["sparse"]

            vector["sparse"] = {
                "indices": to_list(sparse.indices),
                "values": to_list(sparse.values)
            }

        record["vector"] = vector

    return record

OUTPUT_FILE = "ms_marco_top2k_hybrid_backup.json.gz"

offset = None
BATCH_SIZE = 100

with gzip.open(
    OUTPUT_FILE,
    "wt",
    encoding="utf-8"
) as f:

    f.write("[\n")

    first = True

    while True:

        points, offset = client.scroll(
            collection_name=COLLECTION_NAME,
            limit=BATCH_SIZE,
            offset=offset,
            with_payload=True,
            with_vectors=True
        )

        if not points:
            break

        for point in points:

            record = {
                "id": point.id,
                "vector": point.vector,
                "payload": point.payload
            }

            record = make_json_serializable(record)

            if not first:
                f.write(",\n")

            json.dump(
                record,
                f,
                ensure_ascii=False
            )

            first = False

        print("Exported batch")

        if offset is None:
            break

    f.write("\n]")
    
print(f"Saved to {OUTPUT_FILE}")

Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported batch
Exported b

# Load

In [7]:
from qdrant_client import QdrantClient, models

new_client = QdrantClient(
    host="localhost",
    port=6333
)

NEW_COLLECTION = "ms_marco_hybrid"


dense_size = DENSE_DIM


new_client.create_collection(
    collection_name=NEW_COLLECTION,

    vectors_config={
        "dense": models.VectorParams(
            size=dense_size,
            distance=models.Distance.COSINE
        )
    },

    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    }
)

True

In [8]:
import json
import gzip

from qdrant_client import models


def json_to_point(record):

    sparse_data = record["vector"]["sparse"]

    sparse_vector = models.SparseVector(
        indices=sparse_data["indices"],
        values=sparse_data["values"]
    )

    return models.PointStruct(
        id=record["id"],

        vector={
            "dense": record["vector"]["dense"],
            "sparse": sparse_vector
        },

        payload=record["payload"]
    )

In [9]:
BACKUP_FILE = "ms_marco_top2k_hybrid_backup.json.gz"

BATCH_SIZE = 50

with gzip.open(
    BACKUP_FILE,
    "rt",
    encoding="utf-8"
) as f:

    records = json.load(f)

print("Total records:", len(records))

for start in range(0, len(records), BATCH_SIZE):

    batch = records[start:start + BATCH_SIZE]

    qdrant_points = [
        json_to_point(record)
        for record in batch
    ]

    new_client.upsert(
        collection_name=NEW_COLLECTION,
        points=qdrant_points,
        wait=True
    )

    print(
        f"Imported {min(start + BATCH_SIZE, len(records))}"
        f"/{len(records)}"
    )

Total records: 16396
Imported 50/16396
Imported 100/16396
Imported 150/16396
Imported 200/16396
Imported 250/16396
Imported 300/16396
Imported 350/16396
Imported 400/16396
Imported 450/16396
Imported 500/16396
Imported 550/16396
Imported 600/16396
Imported 650/16396
Imported 700/16396
Imported 750/16396
Imported 800/16396
Imported 850/16396
Imported 900/16396
Imported 950/16396
Imported 1000/16396
Imported 1050/16396
Imported 1100/16396
Imported 1150/16396
Imported 1200/16396
Imported 1250/16396
Imported 1300/16396
Imported 1350/16396
Imported 1400/16396
Imported 1450/16396
Imported 1500/16396
Imported 1550/16396
Imported 1600/16396
Imported 1650/16396
Imported 1700/16396
Imported 1750/16396
Imported 1800/16396
Imported 1850/16396
Imported 1900/16396
Imported 1950/16396
Imported 2000/16396
Imported 2050/16396
Imported 2100/16396
Imported 2150/16396
Imported 2200/16396
Imported 2250/16396
Imported 2300/16396
Imported 2350/16396
Imported 2400/16396
Imported 2450/16396
Imported 2500/16396

In [10]:
collection_info = new_client.get_collection(
    NEW_COLLECTION
)

print(collection_info.points_count)

16396


In [11]:
test_points, _ = new_client.scroll(
    collection_name=NEW_COLLECTION,
    limit=1,
    with_payload=True,
    with_vectors=True
)

print(test_points[0])

id='00046cc5-1691-4184-8c31-3dfdb921ce16' payload={'query_id': 11188, 'chunk_id': '11188_2', 'query': 'what part of the brain controls blinking', 'text': "Medulla Oblongata-A part of the brainstem that regulates breathing, heartbeat, and blood flow. Memory-An amazing function of your brain that scientists are still trying to understand. When you remember something, it's not like finding a snapshot in your brain.", 'answer': ['Brain stem'], 'url': 'http://morphonix.com/software/education/science/brain/game/brainarium/brainarium_glossary.html', 'query_type': 'description', 'wellFormedAnswers': []} vector={'sparse': SparseVector(indices=[23249545, 28501148, 79526030, 166093682, 200385224, 366928855, 385736129, 446319932, 499924168, 512480045, 763218840, 806976768, 859745215, 862607732, 1031134330, 1203667988, 1768417799, 1895683639, 1903244910, 1986137636, 2114560765, 2142141949], values=[1.5932107, 1.5932107, 1.5932107, 1.5932107, 1.5932107, 1.5932107, 1.5932107, 1.5932107, 1.5932107, 1.

In [ ]:
# import json 
# with open("ms_marco_top2k.json",'r') as f:
#     data = json.load(f)
#     result = {"documents":[]}
#     for count,d in enumerate(data['documents']):

#         query_id = d['query_id']

        
#         for i, text in enumerate(d['passages']['passage_text']):
#             tmp_data = {"query_id":d['query_id']}
#             tmp_data['chunk_id'] = f"{query_id}_{i}"
#             tmp_data['chunk_text'] = text
#             result['documents'].append(tmp_data)
#         if count >50:
#             break
#     with open("sample_chunks_50.json","w") as f:
#         json.dump(result,f)